# Train sáu OCR trên PC RTX 5070

Tên file được giữ để không làm mất liên kết. Luồng hiện tại train CCT-S, LPRNet và bốn Paddle DET/REC. **VietLPR_v1 đã chốt không train lại** (trước đây có tên VietLPR_v2). Kernel **VN-LPR PC training**. Chạy `scripts/setup_pc.ps1` và nạp `current_baselines.zip` trước. Xem [hướng dẫn PC](../docs/pc_training.md).

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / "configs/ocr/adaptation.json").is_file():
    ROOT = ROOT.parent
assert (ROOT / "configs/ocr/adaptation.json").is_file(), "Mở notebook từ dự án"
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from ocr.adaptation.notebook import run
from ocr.adaptation.common import settings, directory
print(ROOT)


## 1. Kiểm dữ liệu và model giữ nguyên
Chỉ tạo đầu vào train/val; không đọc test để chọn trọng số.

In [ ]:
from ocr.adaptation.data import verify_dataset
from ocr.adaptation.common import validate_frozen
print(verify_dataset())
print(validate_frozen("VietLPR_v1")["precision"])
print([m["name"] for m in settings()["models"] if m["train"]])

## 2. Train → chọn best trên val → export ONNX FP32
Bật `RUN_TRAINING=True` trên PC. Chạy lại tiếp tục checkpoint; model đã train/export xong được kiểm rồi bỏ qua. Không chạy trên ThinkPad.

In [ ]:
RUN_TRAINING = False
MODELS = None  # None: đủ sáu model; hoặc ["CCT_S_VN"]
if RUN_TRAINING:
    command = ["-m", "ocr.adaptation.suite", "run"]
    if MODELS: command += ["--model", *MODELS]
    run(ROOT / ".venv-train-torch/Scripts/python.exe", *command)

## 3. Trạng thái
Chỉ benchmark ở notebook 03 sau khi đủ bảy model FP32 đã chốt.

In [ ]:
import json
for model in settings()["models"]:
    base = directory(model["name"])
    print(model["name"], "FP32 sẵn sàng" if (base / "frozen.json").is_file() else "chưa xong")
state = directory() / "training_state.json"
if state.is_file(): print(state.read_text(encoding="utf-8"))